In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install transformers==4.35.2
!pip install tensorflow==2.16.1

  Using cached ml_dtypes-0.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached tensorboard-2.16.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 589.9/589.9 MB 3.8 MB/s eta 0:00:00
Using cached ml_dtypes-0.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.2 MB)
Using cached tensorboard-2.16.2-py3-none-any.whl (5.5 MB)
  Attempting uninstall: ml-dtypes
    Found existing installation: ml_dtypes 0.5.3
    Uninstalling ml_dtypes-0.5.3:
      Successfully uninstalled ml_dtypes-0.5.3
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.19.0
    Uninstalling tensorboard-2.19.0:
      Successfully uninstalled tensorboard-2.19.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.19.1
    Uninstalling tensorflow-2.19.1:
      Successfully uninstalled tensorflow-2.19.1
ERROR: pip's dependency resolver does not currently take into account all the p

In [5]:
!pip install tf-keras

  Using cached tensorflow-2.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tensorboard-2.19.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached ml_dtypes-0.5.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.9 kB)
Using cached tensorflow-2.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (645.0 MB)
Using cached ml_dtypes-0.5.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (4.9 MB)
Using cached tensorboard-2.19.0-py3-none-any.whl (5.5 MB)
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalling ml-dtypes-0.3.2:
      Successfully uninstalled ml-dtypes-0.3.2
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.16.2
    Uninstalling tensorboard-2.16.2:
      Successfully uninstalled tensorboard-2.16.2
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.16.1
    Uninstalling tensorflow-2.

In [6]:
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, TFBertModel
import numpy as np
import pickle

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [7]:
# from tensorflow.data import Dataset
# from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
# from sklearn.utils.class_weight import compute_class_weight
# from transformers import TFBertModel, BertTokenizer
# import tensorflow as tf
# from tensorflow.keras import Model
# from tensorflow.keras.layers import Dense, Dropout
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split

# # ======================== LOAD ========================
# def prepare_data(file_path):
#     df = pd.read_csv(file_path)
#     df = df.dropna(subset=['text', 'label', 'bahasa'])

#     # Tambah prefix bahasa
#     df['text'] = df['bahasa'].apply(lambda x: '[ID]' if x.lower() == 'in' else '[EN]') + ' ' + df['text']

#     texts = df['text'].tolist()
#     labels = df['label'].tolist()
#     stratify_col = df['bahasa'] + '_' + df['label'].astype(str)

#     X_train, X_test, y_train, y_test = train_test_split(
#         texts, labels,
#         test_size=0.2,
#         stratify=stratify_col,
#         random_state=42
#     )
#     return X_train, X_test, y_train, y_test


# # ======================== TOKENIZATION ========================
# def tokenize(texts, tokenizer, max_len=128):
#     return tokenizer(
#         texts,
#         padding=True,
#         truncation=True,
#         max_length=max_len,
#         return_tensors="tf"
#     )

# # ======================== WARMUP CALLBACK ========================
# class WarmupUnfreezeCallback(Callback):
#     def __init__(self, bert_model, unfreeze_epoch=3):
#         super().__init__()
#         self.bert_model = bert_model
#         self.unfreeze_epoch = unfreeze_epoch

#     def on_epoch_begin(self, epoch, logs=None):
#         if epoch == self.unfreeze_epoch:
#             print(f"\n[INFO] Unfreezing all BERT layers at epoch {epoch}")
#             for layer in self.bert_model.layers:
#                 layer.trainable = True

# # ======================== MODEL ========================
# class ToxicBertClassifier(Model):
#     def __init__(self, dropout_rate=0.3):
#         super().__init__()
#         self.bert = TFBertModel.from_pretrained("bert-base-multilingual-cased")
#         for layer in self.bert.layers:
#             layer.trainable = False  # freeze all first, will be unfrozen later
#         self.dropout = Dropout(dropout_rate)
#         self.dense1 = Dense(256, activation='gelu')
#         self.dense2 = Dense(64, activation='gelu')
#         self.out = Dense(2, activation='softmax')

#     def call(self, inputs, training=False):
#         outputs = self.bert(
#             input_ids=inputs['input_ids'],
#             attention_mask=inputs['attention_mask'],
#             training=training
#         )
#         pooled_output = outputs.last_hidden_state[:, 0]
#         x = self.dropout(pooled_output, training=training)
#         x = self.dense1(x)
#         x = self.dropout(x, training=training)
#         x = self.dense2(x)
#         x = self.dropout(x, training=training)
#         return self.out(x)

# # ======================== DATA PIPELINE ========================
# def create_dataset(encodings, labels, batch_size=32):
#     dataset = Dataset.from_tensor_slices((dict(encodings), labels))
#     return dataset.shuffle(1000).batch(batch_size)

# # ======================== TRAINING FUNCTION ========================
# def train_model(file_path, output_model_path="final_model.keras"):
#     X_train, X_test, y_train, y_test = prepare_data(file_path)
#     tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")

#     train_encodings = tokenize(X_train, tokenizer)
#     test_encodings = tokenize(X_test, tokenizer)

#     y_train = np.array(y_train)
#     y_test = np.array(y_test)

#     train_dataset = create_dataset(train_encodings, y_train)
#     test_dataset = create_dataset(test_encodings, y_test)

#     model = ToxicBertClassifier()

#     # Custom Learning Rate Schedule
#     steps_per_epoch = len(y_train) // 32
#     lr_schedule = tf.keras.optimizers.schedules.PolynomialDecay(
#         initial_learning_rate=5e-5,
#         decay_steps=steps_per_epoch * 10,  # 10 epoch decay
#         end_learning_rate=1e-6,
#         power=1.0
#     )

#     optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=0.01)

#     model.compile(optimizer=optimizer,
#                   loss='sparse_categorical_crossentropy',
#                   metrics=['accuracy'])

#     early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
#     checkpoint = ModelCheckpoint("best_model.keras", monitor='val_accuracy',
#                                  save_best_only=True, mode='max')
#     warmup_callback = WarmupUnfreezeCallback(model.bert, unfreeze_epoch=3)

#     # Hitung class weight
#     class_weights = compute_class_weight(
#         class_weight='balanced',
#         classes=np.unique(y_train),
#         y=y_train
#     )
#     class_weight_dict = dict(enumerate(class_weights))

#     model.fit(
#         train_dataset,
#         validation_data=test_dataset,
#         epochs=100,
#         callbacks=[early_stop, checkpoint, warmup_callback],
#         class_weight=class_weight_dict
#     )

#     model.save(output_model_path)
#     return model

# # ======================== EXECUTION ========================
# if __name__ == "__main__":
#     train_model("/content/drive/My Drive/dataset/data_hapus_inggris.csv")


In [8]:
# from tensorflow.data import Dataset
# from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
# from sklearn.utils.class_weight import compute_class_weight
# from transformers import TFBertModel, BertTokenizer
# import tensorflow as tf
# from tensorflow.keras import Model
# from tensorflow.keras.layers import Dense, Dropout
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split

# # ======================== LOAD ========================
# def prepare_data(file_path):
#     df = pd.read_csv(file_path)
#     df = df.dropna(subset=['text', 'label', 'bahasa'])

#     # Tambah prefix bahasa
#     df['text'] = df['bahasa'].apply(lambda x: '[ID]' if x.lower() == 'in' else '[EN]') + ' ' + df['text']

#     texts = df['text'].tolist()
#     labels = df['label'].tolist()
#     stratify_col = df['bahasa'] + '_' + df['label'].astype(str)

#     X_train, X_test, y_train, y_test = train_test_split(
#         texts, labels,
#         test_size=0.2,
#         stratify=stratify_col,
#         random_state=42
#     )
#     return X_train, X_test, y_train, y_test


# # ======================== TOKENIZATION ========================
# def tokenize(texts, tokenizer, max_len=128):
#     return tokenizer(
#         texts,
#         padding=True,
#         truncation=True,
#         max_length=max_len,
#         return_tensors="tf"
#     )

# # ======================== MODEL ========================
# class ToxicBertClassifier(Model):
#     def __init__(self, dropout_rate=0.3):
#         super().__init__()
#         self.bert = TFBertModel.from_pretrained("bert-base-multilingual-cased")
#         for layer in self.bert.layers:
#             layer.trainable = False  # freeze all first, will be unfrozen later
#         self.dropout = Dropout(dropout_rate)
#         self.dense1 = Dense(256, activation='gelu')
#         self.dense2 = Dense(64, activation='gelu')
#         self.out = Dense(2, activation='softmax')

#     def call(self, inputs, training=False):
#         outputs = self.bert(
#             input_ids=inputs['input_ids'],
#             attention_mask=inputs['attention_mask'],
#             training=training
#         )
#         pooled_output = outputs.last_hidden_state[:, 0]
#         x = self.dropout(pooled_output, training=training)
#         x = self.dense1(x)
#         x = self.dropout(x, training=training)
#         x = self.dense2(x)
#         x = self.dropout(x, training=training)
#         return self.out(x)

# # ======================== DATA PIPELINE ========================
# def create_dataset(encodings, labels, batch_size=32):
#     dataset = Dataset.from_tensor_slices((dict(encodings), labels))
#     return dataset.shuffle(1000).batch(batch_size)

# # ======================== TRAINING FUNCTION ========================
# def train_model(file_path, output_model_path="final_model.keras"):
#     X_train, X_test, y_train, y_test = prepare_data(file_path)
#     tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")

#     train_encodings = tokenize(X_train, tokenizer)
#     test_encodings = tokenize(X_test, tokenizer)

#     y_train = np.array(y_train)
#     y_test = np.array(y_test)

#     train_dataset = create_dataset(train_encodings, y_train)
#     test_dataset = create_dataset(test_encodings, y_test)

#     model = ToxicBertClassifier()

#     # Custom Learning Rate Schedule
#     steps_per_epoch = len(y_train) // 32
#     lr_schedule = tf.keras.optimizers.schedules.PolynomialDecay(
#         initial_learning_rate=5e-5,
#         decay_steps=steps_per_epoch * 10,  # 10 epoch decay
#         end_learning_rate=1e-6,
#         power=1.0
#     )

#     optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=0.01)

#     model.compile(optimizer=optimizer,
#                   loss='sparse_categorical_crossentropy',
#                   metrics=['accuracy'])

#     early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
#     checkpoint = ModelCheckpoint("best_model.keras", monitor='val_accuracy',
#                                  save_best_only=True, mode='max')

#     # Hitung class weight
#     class_weights = compute_class_weight(
#         class_weight='balanced',
#         classes=np.unique(y_train),
#         y=y_train
#     )
#     class_weight_dict = dict(enumerate(class_weights))

#     model.fit(
#         train_dataset,
#         validation_data=test_dataset,
#         epochs=100,
#         callbacks=[early_stop, checkpoint],
#         class_weight=class_weight_dict
#     )

#     model.save(output_model_path)
#     return model

# # ======================== EXECUTION ========================
# if __name__ == "__main__":
#     train_model("/content/drive/My Drive/dataset/5datasentimen_all_clean_ing.csv")


In [9]:
from tensorflow.data import Dataset
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from sklearn.utils.class_weight import compute_class_weight
from transformers import TFBertModel, BertTokenizer
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense, Dropout
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ======================== LOAD ========================
def prepare_data(file_path):
    df = pd.read_csv(file_path)
    df = df.dropna(subset=['text', 'label', 'bahasa'])

    # Tambah prefix bahasa
    df['text'] = df['bahasa'].apply(lambda x: '[ID]' if x.lower() == 'in' else '[EN]') + ' ' + df['text']

    texts = df['text'].tolist()
    labels = df['label'].tolist()
    stratify_col = df['bahasa'] + '_' + df['label'].astype(str)

    X_train, X_test, y_train, y_test = train_test_split(
        texts, labels,
        test_size=0.2,
        stratify=stratify_col,
        random_state=42
    )
    return X_train, X_test, y_train, y_test


# ======================== TOKENIZATION ========================
def tokenize(texts, tokenizer, max_len=128):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="tf"
    )

# ======================== MODEL ========================
class ToxicBertClassifier(Model):
    def __init__(self, dropout_rate=0.3):
        super().__init__()
        self.bert = TFBertModel.from_pretrained("bert-base-multilingual-cased")
        for layer in self.bert.layers:
            layer.trainable = False  # freeze all first, will be unfrozen later
        self.dropout = Dropout(dropout_rate)
        self.dense1 = Dense(256, activation='gelu')
        self.dense2 = Dense(64, activation='gelu')
        self.out = Dense(2, activation='softmax')

    def call(self, inputs, training=False):
        outputs = self.bert(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            training=training
        )
        pooled_output = outputs.last_hidden_state[:, 0]
        x = self.dropout(pooled_output, training=training)
        x = self.dense1(x)
        x = self.dropout(x, training=training)
        x = self.dense2(x)
        x = self.dropout(x, training=training)
        return self.out(x)

# ======================== DATA PIPELINE ========================
def create_dataset(encodings, labels, batch_size=32):
    dataset = Dataset.from_tensor_slices((dict(encodings), labels))
    return dataset.shuffle(1000).batch(batch_size)

# ======================== SAVE ARTIFACTS ========================
def save_artifacts(label_encoder, tokenizer, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    # Save label encoder
    with open(os.path.join(save_dir, "label_encoder.pkl"), "wb") as f:
        pickle.dump(label_encoder, f)

    # Save tokenizer
    tokenizer.save_pretrained(save_dir)

    print(f"[INFO] Artifacts saved to {save_dir}")


# ======================== CUSTOM CALLBACK ========================
class CustomCallback(Callback):
    def __init__(self, model, tokenizer, label_encoder, base_dir, best_val_acc=0.50):
        super().__init__()
        self.model_ref = model
        self.tokenizer = tokenizer
        self.label_encoder = label_encoder

        # Path penyimpanan
        self.save_dir_pb = os.path.join(base_dir, "pb")
        self.save_dir_tflite = os.path.join(base_dir, "tflite")
        self.save_dir_keras = os.path.join(base_dir, "keras")
        self.save_dir_h5 = os.path.join(base_dir, "h5")
        self.save_dir_artifacts = os.path.join(base_dir, "artifacts")

        os.makedirs(self.save_dir_pb, exist_ok=True)
        os.makedirs(self.save_dir_tflite, exist_ok=True)
        os.makedirs(self.save_dir_keras, exist_ok=True)
        os.makedirs(self.save_dir_h5, exist_ok=True)
        os.makedirs(self.save_dir_artifacts, exist_ok=True)

        self.best_val_acc = best_val_acc

    def on_epoch_end(self, epoch, logs=None):
        val_acc = logs.get('val_accuracy')
        train_acc = logs.get('accuracy')

        if val_acc:
            acc_str_val = f"{val_acc:.3f}"
            acc_str_train = f"{train_acc:.3f}"

            if val_acc > self.best_val_acc:
                name = f"epoch_{epoch+1}_accuracyval_{acc_str_val}_accuracytrain_{acc_str_train}"
                self.best_val_acc = val_acc + 0.05
                print("Hasil validasi accuracy terbaik ditemukan, menyimpan model...")

                # Save .pb
                tf.saved_model.save(self.model_ref, os.path.join(self.save_dir_pb, name))

                # Save .keras
                keras_path = os.path.join(self.save_dir_keras, f"{name}.keras")
                self.model_ref.save(keras_path)

                # Save .h5
                h5_path = os.path.join(self.save_dir_h5, f"{name}.h5")
                self.model_ref.save(h5_path)

                # Save tflite
                def model_fn(input_ids, attention_mask):
                    return self.model_ref({'input_ids': input_ids, 'attention_mask': attention_mask}, training=False)

                concrete_func = tf.function(model_fn).get_concrete_function(
                    tf.TensorSpec(shape=[None, None], dtype=tf.int32, name='input_ids'),
                    tf.TensorSpec(shape=[None, None], dtype=tf.int32, name='attention_mask')
                )

                converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
                tflite_model = converter.convert()
                with open(os.path.join(self.save_dir_tflite, f"{name}.tflite"), "wb") as f:
                    f.write(tflite_model)

                # Save artifacts
                # save_artifacts(self.label_encoder, self.tokenizer, self.save_dir_artifacts)
                # print(f"[INFO] Saved model + artifacts: {name}")

            # Stop training bila target tercapai
            if val_acc >= 0.85 and train_acc >= 0.80:
                print("[INFO] Target akurasi tercapai. Training dihentikan.")
                self.model.stop_training = True

# ======================== TRAINING FUNCTION ========================
def train_model(file_path, base_dir="/content/drive/My Drive/colab/saved_model", output_model_path="final_model.keras"):
    X_train, X_test, y_train, y_test = prepare_data(file_path)
    tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")

    train_encodings = tokenize(X_train, tokenizer)
    test_encodings = tokenize(X_test, tokenizer)

    y_train = np.array(y_train)
    y_test = np.array(y_test)

    train_dataset = create_dataset(train_encodings, y_train)
    test_dataset = create_dataset(test_encodings, y_test)

    model = ToxicBertClassifier()

    # Custom Learning Rate Schedule
    steps_per_epoch = len(y_train) // 32
    lr_schedule = tf.keras.optimizers.schedules.PolynomialDecay(
        initial_learning_rate=5e-5,
        decay_steps=steps_per_epoch * 10,  # 10 epoch decay
        end_learning_rate=1e-6,
        power=1.0
    )

    optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=0.01)

    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    checkpoint = ModelCheckpoint("best_model.keras", monitor='val_accuracy',
                                 save_best_only=True, mode='max')

    # Label encoder (dummy → misalnya mapping int to int lagi, tapi harus sama saat inference)
    label_encoder = {i: i for i in np.unique(y_train)}

    # Save artifacts sekali di awal
    save_artifacts(label_encoder, tokenizer, os.path.join(base_dir, "artifacts"))

    custom_cb = CustomCallback(
        model=model,
        tokenizer=tokenizer,
        label_encoder=label_encoder,
        base_dir=base_dir,
        best_val_acc=0.45
    )

    # Hitung class weight
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train),
        y=y_train
    )
    class_weight_dict = dict(enumerate(class_weights))

    model.fit(
        train_dataset,
        validation_data=test_dataset,
        epochs=100,
        callbacks=[early_stop, checkpoint],
        class_weight=class_weight_dict
    )

    model.save(output_model_path)
    return model

# ======================== EXECUTION ========================
if __name__ == "__main__":
    train_model("/content/drive/My Drive/dataset/5datasentimen_all_clean_ing.csv")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


TypeError: 'NoneType' object is not callable